# Preparing CRSP–Compustat Data

The return on a stock and the accounts of its company arrive in different datasets, at different frequencies, under different identifiers. A characteristic portfolio requires a dated connection between them. This notebook prepares that connection from original local WRDS exports. It uses no live WRDS session and no companion-project intermediate.

The workflow deliberately separates **reading files**, **cleaning and linking records**, and **financial measurement**. Notebook [01](01_firm_characteristics.ipynb) measures characteristics; [02](02_portfolio_formation.ipynb) forms portfolios; [03](03_factor_construction.ipynb) constructs traded factors. The reference replication supplies methodological conventions, not runtime data.

Licensed observations and all derived caches remain local. Run this notebook first, then 01–03. A stale source-hash check prevents later notebooks from silently using an old panel.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from factor_asset_pricing.wrds_data import (
    SOURCE_FILES, REQUIRED_COLUMNS, read_wrds_source, source_manifest, save_prepared_data,
)
from factor_asset_pricing.data_preparation import (
    clean_crsp, clean_compustat, clean_ccm, link_crsp_compustat,
    attach_annual_accounts, prepare_risk_free, validate_prepared_panel,
)
RAW = ROOT / "data/raw/wrds"
PROCESSED = ROOT / "data/processed"
plt.style.use("seaborn-v0_8-whitegrid")
diagnostics = {}


## 1. Required original exports

| Source | Required observations and units | Role |
| --- | --- | --- |
| CRSP monthly securities | PERMNO, date, SHRCD, EXCHCD, RET, DLRET, PRC, SHROUT | Security histories; returns are decimal simple returns, prices dollars, shares thousands |
| Compustat annual fundamentals | GVKEY, DATADATE, FYEAR, AT, SEQ, TXDITC, PSTKRV, PSTKL, PSTK, REVT, COGS, XSGA, XINT | Accounts in millions; preferred-stock hierarchy and operating surplus |
| Historical CCM links | GVKEY, LPERMNO, LINKPRIM, LINKTYPE, LINKDT, LINKENDDT | Security/company relationships valid over dated intervals |
| CRSP Treasury export | MCALDT, TMYTM | Annualized yield in percent, converted to a monthly financing return |

For Compustat, INDFMT, DATAFMT, CONSOL and CURCD allow explicit selection of industrial, standard, consolidated USD statements. COSTAT is **not** restricted to active companies. The supplied exports contain those format fields. Without them the filters cannot be independently verified; the cleaner reports their absence.

The registered filenames below match the supplied local exports. Although named `.csv`, these files have an RTF wrapper. The reader strips wrapper controls and row terminators in temporary storage before parsing numeric observations, preserves scientific notation and leading-zero GVKEYs, and never changes the originals. It is not a general rich-text document parser. Required columns are checked before construction.

In [ ]:
source_table = pd.DataFrame([
    {"Dataset": name, "File": filename, "Required variables": ", ".join(REQUIRED_COLUMNS[name]),
     "Present": (RAW / filename).is_file()}
    for name, filename in SOURCE_FILES.items()
]).set_index("Dataset")
display(source_table)
assert source_table.Present.all(), "Place the four original licensed WRDS exports in data/raw/wrds"
manifest = source_manifest(RAW)
display(pd.DataFrame(manifest).T[["file", "bytes", "sha256"]])


## 2. Clean and validate CRSP security months

A PERMNO identifies a security, not a company. We require one observation per security/calendar month and retain historical common-stock share codes 10/11 on NYSE, AMEX and Nasdaq (exchange codes 1/2/3). Filtering applies to records rather than to every date of an identifier. Missing returns remain missing, and firms are not screened using future survival.

Negative PRC is a quotation flag. Existing library functions calculate market equity and delisting-adjusted simple returns:

$$ME_{i,t}=|PRC_{i,t}|SHROUT_{i,t},\qquad R^{adj}_{i,t}=(1+RET_{i,t})(1+DLRET_{i,t})-1.$$

When only RET or DLRET is observed, that observation is retained. Recognized CRSP missing flags/sentinels are removed, but an unknown numeric code or a simple return below −100% raises. No missing delisting payoff is imputed. Missing or nonpositive market equity is visible for later eligibility checks, rather than becoming zero capitalization.

Monthly records use calendar month-start labels; they do not claim that monthly returns were observed at the start of the month.

In [ ]:
crsp = clean_crsp(read_wrds_source(RAW, "crsp"))
diagnostics["crsp"] = crsp.attrs["diagnostics"]
display(pd.Series(diagnostics["crsp"], name="CRSP diagnostics"))
assert not crsp.duplicated(["permno", "mdate"]).any()
assert crsp.exchcd.isin([1, 2, 3]).all() and crsp.shrcd.isin([10, 11]).all()
assert crsp.retadj.dropna().ge(-1).all()
annual_counts = crsp.groupby(crsp.mdate.dt.year).permno.nunique()
ax = annual_counts.plot(figsize=(9, 3.4), color="#2878A5")
ax.set(title="Observed eligible securities by calendar year", xlabel="Year", ylabel="Distinct PERMNOs")
plt.show()


## 3. Clean annual Compustat fundamentals

The annual table keeps one latest DATADATE per GVKEY/FYEAR after format selection and exact-duplicate removal. Conflicting statements at the same final date raise; the workflow does not select an arbitrary accounting version. All original annual histories remain separate from monthly carriers so annual investment can require a genuinely consecutive fiscal-year pair.

The file lacks historical filing dates. We therefore retain the replication's conservative calendar convention: statements dated in calendar year $t-1$ become holding inputs from July $t$ through June $t+1$. The June decision precedes the July holding return. This is an assumed availability schedule, not a point-in-time filing audit. Revised accounts can still differ from what investors actually saw.

The reference characteristic convention uses FYEAR+1 as formation year. When FYEAR differs from DATADATE's calendar year, a fiscal label must not permit an account from a later calendar year to enter an earlier formation. Notebook 01 and the independent-annual path enforce that guard.

In [ ]:
annual = clean_compustat(read_wrds_source(RAW, "compustat"))
diagnostics["compustat"] = annual.attrs["diagnostics"]
display(pd.Series(diagnostics["compustat"], name="Annual-account diagnostics"))
assert not annual.duplicated(["gvkey", "fyear"]).any()
assert annual.available_from.dt.month.eq(7).all()
assert annual.available_from.gt(annual.datadate).all()
# Dated examples show the assumption, not a verified reporting event.
display(annual[["gvkey", "fyear", "datadate", "formation_date", "available_from", "available_until"]].head(5))


## 4. Link through historical CCM intervals

A historical link connects a security to a company only during its recorded interval. The retained link types are LU, LC, LN and LS, with primary indicators P/C, matching the reference. Missing/.B starts and missing/.E ends represent open boundaries. Malformed dates are not silently interpreted as open links.

Evaluate links at the security-month **start label**, preserving the reference boundary convention. A link beginning midmonth therefore first applies at the next month-start label. If several links apply, prefer P over C, LU over LC over LN over LS, then the latest LINKDT. Equally preferred conflicting company identifiers raise. One company may have several securities; one security-month must have only one selected company.

Unlinked stocks are excluded, as in the reference. Consequently the market proxy in 03 is the eligible **linked** equity universe, not the full official French market. We retain this scope to avoid quietly changing the investment exercise.

In [ ]:
links = clean_ccm(read_wrds_source(RAW, "ccm"))
diagnostics["ccm"] = links.attrs["diagnostics"]
display(links.groupby(["linkprim", "linktype"]).size().rename("Links").to_frame())
linked = link_crsp_compustat(crsp, links)
diagnostics["linking"] = linked.attrs["diagnostics"]
display(pd.Series(diagnostics["linking"], name="Link diagnostics"))
assert linked.mdate.between(linked.linkdt, linked.linkenddt).all()
assert not linked.duplicated(["permno", "mdate"]).any()
del crsp, links
gc.collect()


## 5. Accounting timing and the June decision

Monthly accounting carriers are attached with a backward as-of join on GVKEY and their assumed July availability. A carrier expires after the following June; absent accounts are not carried indefinitely. The separate annual table supplies June formation inputs directly, so we do not need a future July row to recover them.

| Quantity at June $t$ | Dated input |
| --- | --- |
| Formation size | June $t$ market equity |
| Book-to-market denominator | December $t-1$ market equity, converted to millions |
| Book equity and profitability | Fiscal $t-1$ statement eligible by the July $t$ convention |
| Investment | Fiscal $t-1$ versus fiscal $t-2$ assets, with consecutive years |
| Portfolio membership | June assignment applied July $t$–June $t+1$ |
| Monthly value weight | Market equity from exactly the preceding calendar month |

Membership is annual; market-value weights update monthly. Neither a missing month nor a missing fiscal year is an intentional extra lag. Original fiscal dates and selected CCM intervals remain in the panel so these restrictions can be audited.

In [ ]:
panel = attach_annual_accounts(linked, annual)
diagnostics["accounting"] = panel.attrs["diagnostics"]
display(pd.Series(diagnostics["accounting"], name="Accounting-window diagnostics"))
del linked
gc.collect()
risk_free = prepare_risk_free(read_wrds_source(RAW, "risk_free"))
diagnostics["risk_free"] = risk_free.attrs["diagnostics"]
panel = panel.merge(risk_free, on="mdate", how="left", validate="many_to_one")


The reference Treasury conversion is

$$RF_t=\exp\left(\frac{TMYTM_t}{100}\frac{30}{365}\right)-1.$$

We retain its fixed 30-day horizon and average available instrument observations within each month. This is a yield-based proxy, not the realized holding-period return of a Treasury bill. The source contains undated instrument placeholders; only records with neither a date nor a yield are removed. Monthly RF coverage must be complete for the linked panel. The official French RF has a different source definition, so MKT differences can reflect financing as well as the stock universe.

## 6. Validate the security-month panel

Diagnostics separate missing returns, capitalization and accounts. Missing accounting values do not delete the stock history; characteristic eligibility is decided later. The final panel retains only common-stock exchange eligibility, valid historical links and correctly dated accounting carriers. It does not infer firm names, reporting dates, company-level share-class aggregation or omitted delisting returns.

The reference RTF reader can turn suffixed CCM end dates into missing dates and therefore open-ended links. This reader removes the wrapper before date parsing. Restoring actual endpoints may reduce coverage; such differences are reported rather than treated as a target to force away.

In [ ]:
diagnostics["panel"] = validate_prepared_panel(panel)
display(pd.Series(diagnostics["panel"], name="Final panel diagnostics"))
missing = panel[["retadj", "me", "seq", "revt", "cogs", "at", "RF"]].isna().mean().mul(100)
display(missing.rename("Missing observations (%)").to_frame())
coverage = panel.groupby(panel.mdate.dt.year).agg(securities=("permno", "nunique"), companies=("gvkey", "nunique"))
ax = coverage.plot(figsize=(9, 3.4), color=["#2878A5", "#168A86"])
ax.set(title="Linked security and company coverage", xlabel="Year", ylabel="Distinct identifiers")
plt.show()


## 7. Save a reproducible local intermediate

The original exports are unchanged. We save the linked monthly panel and independent annual accounts under `data/processed`, together with source hashes, preparation version and aggregate diagnostics. Hash verification in 01–03 rejects stale caches. Run this notebook again whenever an original export or preparation convention changes.

Parquet and executed notebook outputs contain restricted data and stay local under the repository's ignore rules. The source notebooks are stored without outputs; the calculations and figure code remain reproducible. No companion intermediate is read. A fresh machine needs its own licensed exports and a Parquet engine (already included in the notebook extra).

This compact preparation is an educational construction, not an exhaustive French replication. Available sources do not establish historical filing availability or vintage equivalence, and the linked security-level market capitalization convention is retained. Subsequent notebooks compare the computed payoffs with the published French series.

In [ ]:
save_prepared_data(panel, annual, PROCESSED, RAW, diagnostics)
print("Saved local monthly panel, annual accounts, and source-hash manifest.")
print("Next: execute notebooks 01, 02, and 03.")
